
# Hate Speech Dataset Text Preprocessing and Exploratory Data Analysis

This notebook demonstrates the preprocessing of a Hate Speech dataset using Natural Language Processing (NLP) techniques in Python.

## Objectives
The notebook performs:

1. Removing HTML Tags & Special Characters  
2. Handling Whitespace & Extra Spaces  
3. Tokenization  
4. Normalization:
   - Lowercasing
   - Stemming
   - Lemmatization  
5. Stopword Removal  
6. Handling Misspellings & Typos  
7. Removing Duplicates  

It also includes Exploratory Data Analysis (EDA) and visualizations such as:

- Word Clouds
- Histograms
- Frequency Plots
- Text Length Distributions


In [ ]:

# Install required libraries (run if necessary)

# !pip install pandas numpy matplotlib seaborn nltk wordcloud autocorrect beautifulsoup4


In [ ]:

# Import libraries

import pandas as pd
import numpy as np
import re
import string

import matplotlib.pyplot as plt
import seaborn as sns

from bs4 import BeautifulSoup
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

from autocorrect import Speller

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

plt.rcParams['figure.figsize'] = (10,6)


In [ ]:

# Load dataset

df = pd.read_csv('OmbuiHSRaw.csv')

print("Dataset Shape:", df.shape)
df.head()



## Initial Data Inspection


In [ ]:

# Check dataset information

df.info()

print("\nMissing Values:")
print(df.isnull().sum())



## Data Cleaning Preparation

The dataset appears to contain all information in a single column.
We first extract the actual tweet text.


In [ ]:

# Preview raw text

df['tweet'].head()


In [ ]:

# Extract tweet text using regex

df['clean_tweet'] = df['tweet'].astype(str).str.extract(r'"(.*?)"')

df[['tweet', 'clean_tweet']].head()



# Exploratory Data Analysis (EDA)


In [ ]:

# Remove missing extracted tweets

df = df.dropna(subset=['clean_tweet'])

print("Remaining rows:", len(df))


In [ ]:

# Distribution of text lengths

df['text_length'] = df['clean_tweet'].apply(len)

sns.histplot(df['text_length'], bins=50)
plt.title("Distribution of Tweet Lengths")
plt.xlabel("Tweet Length")
plt.ylabel("Frequency")
plt.show()


In [ ]:

# Most common words before preprocessing

all_text = " ".join(df['clean_tweet'].astype(str))

wordcloud = WordCloud(
    width=1000,
    height=500,
    background_color='white'
).generate(all_text)

plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Word Cloud Before Preprocessing")
plt.show()



# Text Preprocessing



## 1. Remove HTML Tags & Special Characters


In [ ]:

def remove_html_special(text):

    # Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove mentions and hashtags symbols only
    text = re.sub(r'@\w+', '', text)
    text = text.replace('#', '')

    # Remove special characters and digits
    text = re.sub(r'[^A-Za-z\s]', '', text)

    return text

df['processed_text'] = df['clean_tweet'].apply(remove_html_special)

df[['clean_tweet', 'processed_text']].head()



## 2. Handle Whitespace & Extra Spaces


In [ ]:

def clean_whitespace(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['processed_text'] = df['processed_text'].apply(clean_whitespace)



## 3. Tokenization


In [ ]:

df['tokens'] = df['processed_text'].apply(word_tokenize)

df[['processed_text', 'tokens']].head()



## 4. Normalization
### Lowercasing, Stemming & Lemmatization


In [ ]:

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def normalize_text(tokens):

    # Lowercase
    tokens = [word.lower() for word in tokens]

    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # Stemming
    tokens = [stemmer.stem(word) for word in tokens]

    return tokens

df['normalized_tokens'] = df['tokens'].apply(normalize_text)

df[['tokens', 'normalized_tokens']].head()



## 5. Stopword Removal


In [ ]:

stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

df['filtered_tokens'] = df['normalized_tokens'].apply(remove_stopwords)

df[['normalized_tokens', 'filtered_tokens']].head()



## 6. Handle Misspellings & Typos


In [ ]:

spell = Speller(lang='en')

def correct_spelling(tokens):
    corrected = [spell(word) for word in tokens]
    return corrected

# Apply on a sample first (full dataset may take time)

df['corrected_tokens'] = df['filtered_tokens'].apply(correct_spelling)

df[['filtered_tokens', 'corrected_tokens']].head()



## 7. Remove Duplicates


In [ ]:

before_duplicates = len(df)

df = df.drop_duplicates(subset=['corrected_tokens'])

after_duplicates = len(df)

print("Rows before duplicate removal:", before_duplicates)
print("Rows after duplicate removal:", after_duplicates)
print("Duplicates removed:", before_duplicates - after_duplicates)



# Final Processed Text


In [ ]:

# Join tokens back into text

df['final_text'] = df['corrected_tokens'].apply(lambda x: " ".join(x))

df[['clean_tweet', 'final_text']].head()



# Visualization After Preprocessing


In [ ]:

# Word cloud after preprocessing

processed_text = " ".join(df['final_text'].astype(str))

wordcloud = WordCloud(
    width=1000,
    height=500,
    background_color='white'
).generate(processed_text)

plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Word Cloud After Preprocessing")
plt.show()


In [ ]:

# Top 20 most frequent words

from collections import Counter

all_words = processed_text.split()

word_freq = Counter(all_words)

common_words = pd.DataFrame(
    word_freq.most_common(20),
    columns=['Word', 'Frequency']
)

sns.barplot(data=common_words, x='Frequency', y='Word')

plt.title("Top 20 Most Frequent Words")
plt.show()



# Summary of Insights from EDA

## Key Findings

1. The dataset contains a large collection of tweets related to social and political discussions.

2. Many tweets included:
   - Hashtags
   - Mentions
   - URLs
   - Special characters
   - Repeated whitespace

3. Commonly occurring words appeared frequently before preprocessing due to:
   - Stopwords
   - Repetitive hashtags
   - Noise in text

4. After preprocessing:
   - The text became cleaner and more standardized.
   - Duplicate and noisy records were removed.
   - Important keywords became more visible in the word cloud.

5. The histogram showed that most tweets were relatively short, which is typical for social media datasets.

6. Word frequency analysis revealed repeated discussion topics and commonly used hate-related or political terms.

## Conclusion

Text preprocessing significantly improves text quality and prepares the dataset for:
- Machine Learning
- Sentiment Analysis
- Hate Speech Detection
- NLP Classification Models


In [ ]:

# Save cleaned dataset

df.to_csv('Cleaned_Hate_Speech_Dataset.csv', index=False)

print("Cleaned dataset saved successfully.")
